# Méthode primitive — filtre à règles (zone de test : Lyon)

Ce notebook montre le filtre À RÈGLES ÉCRITES À LA MAIN de la TOUTE PREMIÈRE méthode essayée par le
projet pour distinguer un tronçon OpenStreetMap cyclable d'un tronçon qui ne l'est pas, testé sur une
petite zone (2 km autour de Lyon) avant tout passage à l'échelle. Ce filtre a été évalué contre une
vérité terrain annotée À LA MAIN sur cette même zone via une webapp dédiée (voir
[`annotation_manuelle/`](annotation_manuelle/) et le README de ce dossier) - la démonstration ci-dessous
ne rejoue que le filtre, pas l'annotation (déjà faite, vendorisée).

**Abandonnée au profit de FOB** (voir `../02_FOB/`) - ce notebook est une démonstration/archive, pas
une étape que la suite du projet réutilise (voir `README.md`).

## 0. Configuration et extraction de la zone de test — [`extraction_zone.py`](extraction_zone.py)

`extraire_zone()` fait une extraction en 2 passes d'un extrait OSM France (celui déjà téléchargé par
l'étape 00) : tous les nœuds dans le rectangle demandé, puis tous les tronçons `highway=*` qui en
touchent au moins un. Le centre (`45.78107, 4.89758`) et le rayon (2 km) sont ceux de l'exploration
d'origine.

In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
sys.path.insert(0, str(ICI))
RACINE = ICI.parents[2]

import pandas as pd
pd.set_option("display.width", 160)

FICHIER_OSM = RACINE / "data" / "donnees_brutes" / "geography" / "osm_reference.osm.pbf"
CENTRE_LAT, CENTRE_LON = 45.78107249550042, 4.897575395740515
RAYON_KM = 2.0

from extraction_zone import bbox_autour_de, extraire_zone

bbox = bbox_autour_de(CENTRE_LAT, CENTRE_LON, RAYON_KM)
print("bbox (lon_min, lat_min, lon_max, lat_max) :", bbox)

troncons = extraire_zone(FICHIER_OSM, bbox)
print(f"{len(troncons):,} tronçons extraits autour de Lyon")
troncons.drop(columns="geometry").head(5)

bbox (lon_min, lat_min, lon_max, lat_max) : (np.float64(4.871739477934781), 45.7630544774824, np.float64(4.923411313546248), 45.799090513518436)
[extraction_zone] passe 1/2 : nœuds dans (np.float64(4.871739477934781), 45.7630544774824, np.float64(4.923411313546248), 45.799090513518436)...


[extraction_zone] 158,026 nœuds trouvés dans la zone
[extraction_zone] passe 2/2 : tronçons touchant la zone...


[extraction_zone] 7,884 tronçons extraits
7,884 tronçons extraits autour de Lyon


,id_osm,highway,bicycle,cycleway,access,maxspeed,surface,name
0,4275143,trunk,NaN,NaN,NaN,70,asphalt,Boulevard Laurent Bonnevay
1,4324344,trunk_link,NaN,NaN,NaN,50,asphalt,NaN
2,4324354,trunk_link,NaN,NaN,NaN,70,NaN,NaN
3,4324377,trunk_link,NaN,NaN,NaN,50,asphalt,NaN
4,4324385,trunk_link,NaN,NaN,NaN,NaN,asphalt,NaN


## 1. Le filtre à règles — [`filtre_regles.py`](filtre_regles.py)

`classifier_troncons()` classe chaque tronçon en 'cyclable' / 'non_cyclable' / 'incertain' à partir de
6 tags seulement (voir sa docstring pour le détail des règles - conservateur : un signal ambigu reste
"incertain", jamais classé au hasard).

In [2]:
from filtre_regles import classifier_troncons

troncons["statut"] = classifier_troncons(troncons)
troncons["statut"].value_counts()

statut
incertain       7211
non_cyclable     374
cyclable         299
Name: count, dtype: int64

## 2. Propagation spatiale — [`propagation_spatiale.py`](propagation_spatiale.py)

Un même itinéraire cyclable est souvent découpé par OSM en plusieurs tronçons aux tags inégaux.
`propager()` reclasse en 'cyclable' un tronçon 'incertain' qui porte le MÊME NOM DE RUE qu'un tronçon
déjà 'cyclable' tout proche - volontairement prudent (jamais par simple contact géométrique).

In [3]:
from propagation_spatiale import propager

troncons_propages = propager(troncons, "statut")
troncons_propages["statut"].value_counts()

[propagation] itération 1 : 82 tronçon(s) incertain(s) -> cyclable


[propagation] itération 2 : 31 tronçon(s) incertain(s) -> cyclable


statut
incertain       7098
cyclable         412
non_cyclable     374
Name: count, dtype: int64

## 3. Ce que ça donne

Avant/après propagation, et un aperçu de ce qui reste "incertain" - typiquement des rues résidentielles
à 30 km/h sans aucun tag vélo, exactement le genre de cas que FOB (`../02_FOB/`) tranche mieux en
apprenant sur de vrais exemples plutôt qu'avec des règles fixes.

In [4]:
avant = troncons["statut"].value_counts()
apres = troncons_propages["statut"].value_counts()
comparaison = pd.DataFrame({"avant_propagation": avant, "apres_propagation": apres}).fillna(0).astype(int)
comparaison["difference"] = comparaison["apres_propagation"] - comparaison["avant_propagation"]
comparaison

,avant_propagation,apres_propagation,difference
statut,,,
cyclable,299,412,113
incertain,7211,7098,-113
non_cyclable,374,374,0


In [5]:
troncons_propages[troncons_propages["statut"] == "incertain"][["highway", "bicycle", "cycleway", "maxspeed", "name"]].head(10)

,highway,bicycle,cycleway,maxspeed,name
6,residential,NaN,NaN,30,Rue Dedieu
7,tertiary,NaN,NaN,50,Boulevard du 11 Novembre 1918
8,tertiary,NaN,NaN,50,Rond-point de La Doua
9,residential,NaN,NaN,NaN,Impasse Anatole France
11,residential,NaN,NaN,NaN,Rue Beauséjour
12,secondary,NaN,NaN,NaN,NaN
13,unclassified,NaN,NaN,NaN,Allée Georges Salendre
14,residential,NaN,NaN,NaN,Rue Javelot
15,residential,NaN,NaN,NaN,Rue Beausite
17,unclassified,NaN,NaN,NaN,Rue Paul Éluard
